# GroupBy in Pandas — Split, Apply, Combine

`groupby()` answers questions of the shape **"per X, what is Y?"** —
average salary per department, survival rate per class, titles per year.

You will learn:

- The split → apply → combine idea
- `mean()`, `count()`, `size()`, `sum()`, `min()`, `max()`, `median()`
- `agg()` with a list and with a dictionary
- Grouping by **two** columns, and `unstack()`
- `transform()` — putting a group result back on every row
- 25 practice questions on the Titanic dataset

Datasets: a small employee table, then `../data/titanic_dataset.csv`

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = {
'Employee': ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
'Department': ['IT', 'HR', 'IT', 'Finance', 'HR', 'Finance', 'IT'],   
'Salary': [50000, 40000, 55000, 60000, 42000, 65000, 58000],
'Experience': [2, 1, 3, 5, 2, 6, 4]
}


df = pd.DataFrame(data)
df

,Employee,Department,Salary,Experience
0,A,IT,50000,2
1,B,HR,40000,1
2,C,IT,55000,3
3,D,Finance,60000,5
4,E,HR,42000,2
5,F,Finance,65000,6
6,G,IT,58000,4


3. What is GroupBy?

groupby() is used to split data into groups, apply a function, and combine results.

Split → Apply → Combine

Basic GroupBy (Single Column)
Group by Department

In [3]:
df.groupby('Department')

In [4]:
df.groupby('Department')['Salary'].mean()
#Calculate Mean Salary per Department

Department
Finance    62500.000000
HR         41000.000000
IT         54333.333333
Name: Salary, dtype: float64

In [5]:
df.groupby("Department")["Employee"].count()

Department
Finance    2
HR         2
IT         3
Name: Employee, dtype: int64

In [6]:
# Multiple Aggregations
df.groupby('Department')['Salary'].agg(['mean', 'max', 'min', 'sum'])

,mean,max,min,sum
Department,,,,
Finance,62500.000000,65000,60000,125000
HR,41000.000000,42000,40000,82000
IT,54333.333333,58000,50000,163000


In [7]:
# GroupBy with Multiple Columns 
df.groupby(['Department', 'Experience'])['Salary'].mean()

Department  Experience
Finance     5             60000.0
            6             65000.0
HR          1             40000.0
            2             42000.0
IT          2             50000.0
            3             55000.0
            4             58000.0
Name: Salary, dtype: float64

In [8]:
df.groupby('Department')['Employee'].count()
#Using GroupBy with Count

Department
Finance    2
HR         2
IT         3
Name: Employee, dtype: int64

In [9]:
def salary_range(x):
 return x.max() - x.min()


df.groupby('Department')['Salary'].apply(salary_range)

Department
Finance    5000
HR         2000
IT         8000
Name: Salary, dtype: int64

---

## GroupBy on a real dataset — Titanic

The employee table above had 7 rows. Everything below is the same set of methods
on 1309 passengers, where the answers actually mean something.

In [10]:
import pandas as pd

df = pd.read_csv("../data/titanic_dataset.csv")
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [11]:
#Average Fare per Passenger Class
df.groupby('Pclass')['Fare'].mean()

Pclass
1    87.508992
2    21.179196
3    13.302889
Name: Fare, dtype: float64

In [12]:
df.groupby('Sex')['Survived'].mean()
#Survival Rate by Gender

Sex
female    0.82618
male      0.12930
Name: Survived, dtype: float64

In [13]:
# death rate by gender
survival_rate = df.groupby('Sex')['Survived'].mean()
death_rate = 1 - survival_rate

print(death_rate)

Sex
female    0.17382
male      0.87070
Name: Survived, dtype: float64


In [14]:
# Multiple Aggregations
df.groupby('Pclass')['Fare'].agg(['mean', 'max', 'min', 'count'])

,mean,max,min,count
Pclass,,,,
1,87.508992,512.3292,0.0,323
2,21.179196,73.5000,0.0,277
3,13.302889,69.5500,0.0,708


In [15]:
#GroupBy Multiple Columns
df.groupby(['Pclass', 'Sex'])['Survived'].mean()

Pclass  Sex   
1       female    0.979167
        male      0.251397
2       female    0.943396
        male      0.099415
3       female    0.666667
        male      0.095335
Name: Survived, dtype: float64

In [16]:
# Average age of survivors vs non-survivors
df.groupby('Survived')['Age'].mean()

Survived
0    30.510986
1    28.931079
Name: Age, dtype: float64

In [17]:
df['Survived'].value_counts()

Survived
0    815
1    494
Name: count, dtype: int64

In [18]:
# Count of male and female passengers per class
df.groupby(['Pclass', 'Sex']).size()

Pclass  Sex   
1       female    144
        male      179
2       female    106
        male      171
3       female    216
        male      493
dtype: int64

In [19]:
# transform() vs agg()
#
#   groupby(...).mean()      -> ONE row per group (a small summary table)
#   groupby(...).transform() -> ONE row per ORIGINAL row (same length as df)
#
# transform is what you want when you need the group value back on every row,
# e.g. to compare each passenger against the average of their own group.
df["average_age"] = df.groupby("Sex")["Age"].transform("mean")
df[["Name", "Sex", "Age", "average_age"]].head()

,Name,Sex,Age,average_age
0,"Braund, Mr. Owen Harris",male,22.0,30.585228
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,28.687088
2,"Heikkinen, Miss. Laina",female,26.0,28.687088
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,28.687088
4,"Allen, Mr. William Henry",male,35.0,30.585228


In [20]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,average_age
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,30.585228
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,28.687088
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,28.687088
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,28.687088
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,30.585228
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,0,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S,30.585228
1305,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,28.687088
1306,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,30.585228
1307,1308,0,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S,30.585228


In [21]:
# Only two distinct values — one per Sex group, repeated down the column
df["average_age"].unique()

array([30.58522796, 28.68708763])

In [22]:
df.groupby("Sex")["Age"].mean()

Sex
female    28.687088
male      30.585228
Name: Age, dtype: float64

In [23]:
# Which embark town has highest survival rate?

df.groupby("Embarked")["Age"].mean()

Embarked
C    32.332170
Q    28.630000
S    29.245205
Name: Age, dtype: float64

---

## Task — 25 GroupBy practice questions (Titanic)

> *Original note: "make a 25 question and practise yourself — first write the question, then answer"*

Each cell states the question as a comment and answers it with one groupby.

In [24]:
# Q1. How many passengers were in each class?
df.groupby("Pclass").size()

Pclass
1    323
2    277
3    709
dtype: int64

In [25]:
# Q2. How many men and how many women?
df.groupby("Sex").size()

Sex
female    466
male      843
dtype: int64

In [26]:
# Q3. Average age per class
df.groupby("Pclass")["Age"].mean()

Pclass
1    39.159930
2    29.506705
3    24.816367
Name: Age, dtype: float64

In [27]:
# Q4. Average fare per class
df.groupby("Pclass")["Fare"].mean()

Pclass
1    87.508992
2    21.179196
3    13.302889
Name: Fare, dtype: float64

In [28]:
# Q5. Survival rate per class.
# Survived is 0/1, so the MEAN of that column is the survival rate.
df.groupby("Pclass")["Survived"].mean()

Pclass
1    0.575851
2    0.422383
3    0.269394
Name: Survived, dtype: float64

In [29]:
# Q6. Survival rate per sex
df.groupby("Sex")["Survived"].mean()

Sex
female    0.82618
male      0.12930
Name: Survived, dtype: float64

In [30]:
# Q7. Survival rate per class AND sex (group by two columns)
df.groupby(["Pclass", "Sex"])["Survived"].mean()

Pclass  Sex   
1       female    0.979167
        male      0.251397
2       female    0.943396
        male      0.099415
3       female    0.666667
        male      0.095335
Name: Survived, dtype: float64

In [31]:
# Q8. The same result as a readable table — unstack() moves the inner level
# ("Sex") up into the columns.
df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack()

Sex,female,male
Pclass,,
1,0.979167,0.251397
2,0.943396,0.099415
3,0.666667,0.095335


In [32]:
# Q9. How many passengers boarded at each port?
df.groupby("Embarked").size()

Embarked
C    270
Q    123
S    914
dtype: int64

In [33]:
# Q10. Survival rate per embarkation port
df.groupby("Embarked")["Survived"].mean().sort_values(ascending=False)

Embarked
C    0.492593
Q    0.439024
S    0.333698
Name: Survived, dtype: float64

In [34]:
# Q11. Oldest and youngest passenger in each class
df.groupby("Pclass")["Age"].agg(["min", "max"])

,min,max
Pclass,,
1,0.92,80.0
2,0.67,70.0
3,0.17,74.0


In [35]:
# Q12. Several statistics at once with a LIST
df.groupby("Pclass")["Fare"].agg(["count", "mean", "median", "min", "max"])

,count,mean,median,min,max
Pclass,,,,,
1,323,87.508992,60.0000,0.0,512.3292
2,277,21.179196,15.0458,0.0,73.5000
3,708,13.302889,8.0500,0.0,69.5500


In [36]:
# Q13. A different function per column, with a DICTIONARY
df.groupby("Pclass").agg({"Fare": "mean", "Age": "median", "Survived": "sum"})

,Fare,Age,Survived
Pclass,,,
1,87.508992,39.0,186
2,21.179196,29.0,117
3,13.302889,24.0,191


In [37]:
# Q14. Total fare collected per class
df.groupby("Pclass")["Fare"].sum()

Pclass
1    28265.4043
2     5866.6374
3     9418.4452
Name: Fare, dtype: float64

In [38]:
# Q15. Median age per sex (less affected by extreme values than the mean)
df.groupby("Sex")["Age"].median()

Sex
female    27.0
male      28.0
Name: Age, dtype: float64

In [39]:
# Q16. count() vs size().
# count() ignores missing values, size() counts every row.
# The gap between them is exactly the number of missing ages.
compare = pd.DataFrame({
    "count_age": df.groupby("Pclass")["Age"].count(),
    "size":      df.groupby("Pclass").size()
})
compare["missing_age"] = compare["size"] - compare["count_age"]
compare

,count_age,size,missing_age
Pclass,,,
1,284,323,39
2,261,277,16
3,501,709,208


In [40]:
# Q17. How many passengers travelled with siblings/spouse, per class?
df.groupby("Pclass")["SibSp"].sum()

Pclass
1    141
2    109
3    403
Name: SibSp, dtype: int64

In [41]:
# Q18. Average family size per class (SibSp + Parch + the passenger themselves)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df.groupby("Pclass")["FamilySize"].mean()

Pclass
1    1.801858
2    1.761733
3    1.968970
Name: FamilySize, dtype: float64

In [42]:
# Q19. Survival rate by family size — does travelling alone help?
df.groupby("FamilySize")["Survived"].mean()

FamilySize
1     0.292405
2     0.531915
3     0.559748
4     0.720930
5     0.227273
6     0.200000
7     0.312500
8     0.125000
11    0.181818
Name: Survived, dtype: float64

In [43]:
# Q20. Number of DIFFERENT ports used per class (nunique counts distinct values)
df.groupby("Pclass")["Embarked"].nunique()

Pclass
1    3
2    3
3    3
Name: Embarked, dtype: int64

In [44]:
# Q21. The most expensive ticket in each class, with the passenger's name.
# idxmax() gives the row LABEL of the maximum, then .loc fetches those rows.
top_fare_rows = df.groupby("Pclass")["Fare"].idxmax()
df.loc[top_fare_rows, ["Pclass", "Name", "Fare"]]

,Pclass,Name,Fare
258,1,"Ward, Miss. Anna",512.3292
72,2,"Hood, Mr. Ambrose Jr",73.5000
159,3,"Sage, Master. Thomas Henry",69.5500


In [45]:
# Q22. Children (<18) vs adults — survival rate.
# Build the label first with .loc, then group by it.
df.loc[df["Age"] < 18, "AgeGroup"] = "Child"
df.loc[df["Age"] >= 18, "AgeGroup"] = "Adult"
df.groupby("AgeGroup")["Survived"].mean()

AgeGroup
Adult    0.380045
Child    0.506494
Name: Survived, dtype: float64

In [46]:
# Q23. Survival rate by age group AND sex
df.groupby(["AgeGroup", "Sex"])["Survived"].mean().unstack()

Sex,female,male
AgeGroup,,
Adult,0.851266,0.121528
Child,0.763889,0.280488


In [47]:
# Q24. A custom function with apply(): the fare RANGE inside each class
def fare_range(x):
    return x.max() - x.min()

df.groupby("Pclass")["Fare"].apply(fare_range)

Pclass
1    512.3292
2     73.5000
3     69.5500
Name: Fare, dtype: float64

In [48]:
# Q25. transform() again: how far is each passenger's fare from their class average?
df["class_avg_fare"] = df.groupby("Pclass")["Fare"].transform("mean")
df["fare_vs_class"] = (df["Fare"] - df["class_avg_fare"]).round(2)
df[["Name", "Pclass", "Fare", "class_avg_fare", "fare_vs_class"]].head(10)

,Name,Pclass,Fare,class_avg_fare,fare_vs_class
0,"Braund, Mr. Owen Harris",3,7.2500,13.302889,-6.05
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,71.2833,87.508992,-16.23
2,"Heikkinen, Miss. Laina",3,7.9250,13.302889,-5.38
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,53.1000,87.508992,-34.41
4,"Allen, Mr. William Henry",3,8.0500,13.302889,-5.25
5,"Moran, Mr. James",3,8.4583,13.302889,-4.84
6,"McCarthy, Mr. Timothy J",1,51.8625,87.508992,-35.65
7,"Palsson, Master. Gosta Leonard",3,21.0750,13.302889,7.77
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",3,11.1333,13.302889,-2.17
9,"Nasser, Mrs. Nicholas (Adele Achem)",2,30.0708,21.179196,8.89


### Key takeaways

| Goal | Code |
|------|------|
| Rows per group | `df.groupby("col").size()` |
| Non-missing values per group | `df.groupby("col")["x"].count()` |
| One statistic | `.mean()` `.sum()` `.median()` `.min()` `.max()` |
| Several statistics | `.agg(["mean", "max"])` |
| Different function per column | `.agg({"Fare": "mean", "Age": "median"})` |
| Group by two columns | `df.groupby(["a", "b"])` |
| Turn the inner level into columns | `.unstack()` |
| Rate of a 0/1 column | `.mean()` on that column |
| Group value back on every row | `.transform("mean")` |
| Custom function | `.apply(my_function)` |

**`count()` vs `size()`:** `size()` counts rows, `count()` counts non-missing values.
The difference tells you how much is missing in each group.

---

## Appendix — combining two tables with `merge()`

Grouping summarises **one** table. When information is split across **two** tables
you join them first with `pd.merge()`. A short recap follows; the full walkthrough
is in **`pandas_joins_simple.ipynb`**.

In [49]:
import pandas as pd

# Two tiny tables that share an "id" column
df1 = pd.DataFrame({'id': [1, 2, 3], 'name':  ['A', 'B', 'C']})
df2 = pd.DataFrame({'id': [1, 2, 4], 'score': [85, 90, 88]})

print(df1)
print()
print(df2)

   id name
0   1    A
1   2    B
2   3    C

   id  score
0   1     85
1   2     90
2   4     88


In [50]:
# merge() lines the two tables up on the shared key.
# Default how='inner' -> only the ids present in BOTH tables (1 and 2).
pd.merge(df1, df2, on='id')

,id,name,score
0,1,A,85
1,2,B,90


In [51]:
# A second example with more realistic column names
students = pd.DataFrame({
    'student_id':   [101, 102, 103, 104],
    'student_name': ['Ram', 'Sita', 'Hari', 'Gita'],
    'department':   ['CS', 'IT', 'CS', 'BCA']
})
students

,student_id,student_name,department
0,101,Ram,CS
1,102,Sita,IT
2,103,Hari,CS
3,104,Gita,BCA


In [52]:
marks = pd.DataFrame({
    'student_id': [101, 102, 105],
    'marks':      [88, 92, 75]
})
marks

,student_id,marks
0,101,88
1,102,92
2,105,75


In [53]:
# INNER — only students who also have marks (101, 102)
pd.merge(students, marks, on='student_id', how='inner')

,student_id,student_name,department,marks
0,101,Ram,CS,88
1,102,Sita,IT,92


In [54]:
# LEFT — every student; marks become NaN where there is no match
pd.merge(students, marks, on='student_id', how='left')

,student_id,student_name,department,marks
0,101,Ram,CS,88.0
1,102,Sita,IT,92.0
2,103,Hari,CS,NaN
3,104,Gita,BCA,NaN


In [55]:
# RIGHT — every marks row; student details become NaN where there is no match
pd.merge(students, marks, on='student_id', how='right')

,student_id,student_name,department,marks
0,101,Ram,CS,88
1,102,Sita,IT,92
2,105,NaN,NaN,75


In [56]:
# OUTER — everything from both tables
pd.merge(students, marks, on='student_id', how='outer')

,student_id,student_name,department,marks
0,101,Ram,CS,88.0
1,102,Sita,IT,92.0
2,103,Hari,CS,NaN
3,104,Gita,BCA,NaN
4,105,NaN,NaN,75.0


### Easy memory trick

| Join | Keeps |
|------|-------|
| **Inner** | common rows only |
| **Left** | everything from the first table |
| **Right** | everything from the second table |
| **Outer** | everything from both |

Full walkthrough, plus `concat()` and the common mistakes: **`pandas_joins_simple.ipynb`**.